문서를 세 개의 class로 분류하는 간단한 두 개의 층을 가진 신경망을 훈련시키면서 이를 설명하겠다.

In [1]:
import re, torch
import torch.nn as nn

torch.manual_seed(42)

corpus와 label 생성

In [2]:
docs=[
  "Movies are fun for everyone.",
  "Watching movies is great fun.",
  "Enjoy a great movie today.",
  "Research is interesting and important.",
  "Learning math is very important.",
  "Science discovery is interesting.",
  "Rock is great to listen to.",
  "Listen to music for fun.",
  "Music is fun for everyone.",
  "Listen to folk music!"
]

labels=[1,1,1,3,3,3,2,2,2,2] #각 문서에 대한 label
num_classes=len(set(labels))

num_classes

3

문서를 BoW 표현으로 변환한다.
이를 위해 먼저 vocalbulary를 만든다.

In [3]:
#입력 text를 소문자 단어로 분할
def tokenize(text):
  return re.findall(r"\w+",text.lower())

#vocabulary 구축
def get_vocabulary(texts):
  tokens={token for text in texts for token in tokenize(text)}
  return {word: idx for idx, word in enumerate(sorted(tokens))}

vocabulary=get_vocabulary(docs)
vocabulary

{'a': 0,
 'and': 1,
 'are': 2,
 'discovery': 3,
 'enjoy': 4,
 'everyone': 5,
 'folk': 6,
 'for': 7,
 'fun': 8,
 'great': 9,
 'important': 10,
 'interesting': 11,
 'is': 12,
 'learning': 13,
 'listen': 14,
 'math': 15,
 'movie': 16,
 'movies': 17,
 'music': 18,
 'research': 19,
 'rock': 20,
 'science': 21,
 'to': 22,
 'today': 23,
 'very': 24,
 'watching': 25}

문서를 vector로 변환하는 함수

In [5]:
#문서와 vocabulary를 받고
#해당 문서의 BoW 표현을 반환한다.
def doc_to_bow(doc,vocabulary):
  tokens=set(tokenize(doc))
  bow=[0]*len(vocabulary)
  for token in tokens:
    if token in vocabulary:
      bow[vocabulary[token]]=1
  return bow

In [7]:
#문서와 label을 숫자로 변환
vectors=torch.tensor(
    [doc_to_bow(doc,vocabulary) for doc in docs],
    dtype=torch.float32
)
#class 1, 2, 3을 index 0, 1, 2로 바꾸기 위해 1을 뺀다.
labels=torch.tensor(labels,dtype=torch.long) - 1

In [8]:
vectors

tensor([[0., 0., 1., 0., 0., 1., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
         0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 1., 0., 0., 0., 0., 1.,
         0., 0., 0., 0., 0., 0., 0., 1.],
        [1., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0.,
         0., 0., 0., 0., 0., 1., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 0., 0., 0., 0., 0.,
         0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 1., 1., 0., 1., 0., 0.,
         0., 0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0.,
         0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0., 1., 0., 0., 0.,
         0., 0., 1., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
         1., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0.,

In [9]:
labels

tensor([0, 0, 0, 2, 2, 2, 1, 1, 1, 1])

위에서 보듯 각 문서의 one-hot encoding과 그 문서에 대한 label이 잘 된 것을 볼 수 있다.

다음으로 모델을 정의하겠다.

In [10]:
input_dim=len(vocabulary)
hidden_dim=50
output_dim=num_classes

class SimpleClassifier(nn.Module):
  def __init__(self, input_dim, hidden_dim,output_dim):
    super().__init__()
    self.fc1=nn.Linear(input_dim,hidden_dim)
    self.relu=nn.ReLU()
    self.fc2=nn.Linear(hidden_dim,output_dim)
  def forward(self,x):
    x=self.fc1(x)
    x=self.relu(x)
    x=self.fc2(x)
    return x

model=SimpleClassifier(input_dim,hidden_dim,output_dim)

In [11]:
model

SimpleClassifier(
  (fc1): Linear(in_features=26, out_features=50, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=50, out_features=3, bias=True)
)

다음으로 손실 함수를 정의하고, 경사 하강법 알고리즘을 선택 및 훈련 루프를 준비하겠다.

In [12]:
#pytorch의 CrossEntropyLoss라는 class 에는 내부적으로 softmax와 cross entropy loss가 결합되어 있다.
criterion=nn.CrossEntropyLoss()
#확률적 경사하강법 사용
optimizer=torch.optim.SGD(model.parameters(),lr=0.001)

for step in range(3000):
  #gradient 초기화하여 이전 batch의 gradient 누적을 방지한다.
  optimizer.zero_grad()
  #model에 dataset을 넣고, cross entropy loss 계산
  loss=criterion(model(vectors),labels)
  #계산된 loss로 backpropagation 실행
  loss.backward()
  #weight 수정
  optimizer.step()



새로운 문서에서 모델을 test한다.

In [13]:
new_docs=[
    "Listening to rock music is fun.",
    "I love science very much."
]
class_names=["Cinema","Music","Science"]

new_doc_vectors=torch.tensor(
    [doc_to_bow(new_doc,vocabulary) for new_doc in new_docs],
    dtype=torch.float32
)

#gradient 추적을 비활성화한다.
#훈련이 아닌 test이기 때문이다.
with torch.no_grad():
  #model이 모든 입력을 동시에 처리한다.
  outputs=model(new_doc_vectors)
  #가장 큰 logit의 index을 찾으며, 이 값은 예측 class에 해당한다.
  #앞서 0 기반 indexing을 위해 1을 뺐던 것을 보상하기 위해 1을 더한다.
  predicted_ids=torch.argmax(outputs,dim=1) + 1

for i, new_doc in enumerate(new_docs):
  print(f"{new_doc}: {class_names[predicted_ids[i].item() - 1]}")


Listening to rock music is fun.: Music
I love science very much.: Science
